# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/28/connecting-my-cou

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/11/11/ai-live-event/
https://edwarddonner.com/2025/11/11/ai-live-event/
https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/
htt

In [8]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'partner page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [12]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [13]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 8 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [14]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 15 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'docs page', 'url': 'https://huggingface.co/docs'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'inference endpoints', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Discuss forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'Status page', 'url':

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [15]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [16]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Tongyi-MAI/Z-Image-Turbo
Updated
14 days ago
•
363k
•
3.29k
Qwen/Qwen-Image-Layered
Updated
4 days ago
•
5.38k
•
539
google/functiongemma-270m-it
Updated
4 days ago
•
14.3k
•
472
nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
Updated
3 days ago
•
86.2k
•
441
XiaomiMiMo/MiMo-V2-Flash
Updated
5 days ago
•
8.83k
•
398
Browse 2M+ models
Spaces
Running
on
Zero
Featured
436
TRELLIS.2
🏢
436
High-fidelity 3D Generation from images
Running
on
Zero
MCP
Featured
339
Chatterbox Turbo Demo
⚡
339
Chatterbox Turbo Demo
Running
on
Zero
758
Z Image Turbo
🖼
758
Generate images from text prompts
Running
on
Zero
Featured
183
Qwen Image Layered
🚀
183
Dec

In [20]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [21]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [22]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nTongyi-MAI/Z-Image-Turbo\nUpdated\n14 days ago\n•\n363k\n•\n3.3k\nQwen/Qwen-Image-Layered\nUpdated\n4 days ago\n•\n5.38k\n•\n541\ngoogle/functiongemma-270m-it\nUpdated\n4 days ago\n•\n14.3k\n•\n475\nnvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16\nUpdated\n3 days ago\n•\n86.2k\n•\n443\nXiaomiMiMo/MiMo-V2-Flash\nUpdated\n5 days ago\n•\n8.83k\n•\n399\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\n

In [23]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [24]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


# Hugging Face Company Brochure

---

## About Hugging Face

Hugging Face is the AI community building the future of machine learning. It is a collaborative platform where developers, researchers, and enterprises come together to create, share, and advance machine learning models, datasets, and applications. Established as the home of machine learning, Hugging Face empowers its users to build better, faster, and more innovative AI solutions.

---

## What We Offer

- **Models:** Access and collaborate on over 2 million open-source machine learning models spanning multiple modalities including text, image, video, audio, and even 3D data.
- **Datasets:** Browse and contribute to a growing repository of over 500,000 datasets to fuel diverse AI use cases.
- **Spaces:** Deploy and explore over 1 million AI-powered applications and demos powered by community and enterprise projects.
- **Community:** Join a vibrant network of AI practitioners sharing knowledge, research papers, and projects. Our active community hosts over 74,000 AI enthusiasts constantly contributing to open datasets, models, and scientific papers.
- **Enterprise Solutions:** Accelerate your team’s AI development with our paid compute services and enterprise-grade platforms designed to provide advanced tools and collaborative environments.
- **Open Source Stack:** Move faster in your AI projects by leveraging Hugging Face’s robust open source ecosystem.

---

## Customer & Community Highlights

Hugging Face serves a wide range of users:
- Independent AI researchers and data scientists building and sharing projects.
- Developers and machine learning engineers advancing AI applications.
- Educational institutions exploring AI research.
- Enterprises seeking powerful AI tools and collaboration platforms, supported by dedicated computing and enterprise solutions.

Trending models and datasets frequently updated by top contributors and AI leaders demonstrate the platform’s dynamic and cutting-edge environment. Popular models include powerful image generation and language understanding systems, all accessible to users worldwide.

---

## Our Culture

Hugging Face thrives on openness, collaboration, and community-driven innovation. The company encourages sharing knowledge freely and building AI tools that benefit all. Our culture reflects:
- **Community First:** Prioritize contributors and users, valuing open exchange and educational resources.
- **Innovation:** Constantly push the boundaries of AI technologies across various modalities.
- **Empowerment:** Provide the tools and infrastructure necessary for everyone — from hobbyists to enterprises — to build and deploy AI at scale.
- **Transparency:** Active publication of research and open sharing of datasets and models.

---

## Careers at Hugging Face

Join the forefront of AI innovation by working with Hugging Face! We offer exciting career opportunities for those passionate about machine learning, open source, and community engagement. Roles typically focus on:
- Machine Learning Research and Engineering
- Software Development
- Community Management and Developer Relations
- Enterprise Solutions and DevOps

By joining the Hugging Face team, you contribute to a mission of democratizing AI and shaping the future of technology worldwide.

---

## Get Involved

- Visit [huggingface.co](https://huggingface.co) to explore models, datasets, and applications.
- Sign up for free to start sharing your own projects or collaborate with others.
- Upgrade to paid Compute and Enterprise plans for professional and team-level capabilities.
- Follow the community’s latest research, demos, and updates.

---

**Hugging Face** — *Where the machine learning community builds the future together.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [25]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [26]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 19 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is a global AI community and collaboration platform dedicated to building the future of machine learning (ML). It serves as the central hub where ML practitioners—from engineers, scientists to end users—can share, explore, and collaborate on cutting-edge ML models, datasets, and applications. Through its open and ethical AI initiatives, Hugging Face is empowering the next generation of AI innovation.

---

## The Platform

- **Models:** Access over 2 million open-source machine learning models across all modalities including text, image, video, audio, and 3D. From image generation to natural language processing, models are regularly updated and curated by the community.

- **Datasets:** Browse and contribute to more than 500,000 datasets, supporting research and application development in a wide variety of domains including medical, scientific, and multimedia data.

- **Spaces:** Host and deploy AI applications easily with Hugging Face Spaces. Users can experiment with AI apps, from high-fidelity 3D image generators to interactive demos that run on zero or minimal computing resources.

- **Community:** A vibrant and fast-growing community fuels collaboration and knowledge sharing. Users can build their machine learning portfolio and engage with peers through the platform.

- **Enterprise Solutions:** Hugging Face offers advanced paid Compute and Enterprise services designed for teams and organizations to accelerate AI development with robust infrastructure and professional support.

---

## Company Culture

Hugging Face fosters an open, collaborative, and inclusive culture centered around transparency and shared growth. It promotes:

- **Open Source Ethos:** Commitment to open and ethical AI development.
- **Collaboration:** Enabling ML practitioners worldwide to work together seamlessly.
- **Innovation:** Encouraging experimentation with the latest AI technologies.
- **Education:** Empowering users to learn, share and grow their ML expertise.
- **Community-Driven:** Active engagement with contributors and users to advance AI inclusively.

---

## Customers & Users

- Machine Learning Researchers and Practitioners
- AI Engineers and Developers
- Academic Institutions
- Enterprises looking to integrate AI at scale
- Startups building AI-powered applications
- Data Scientists and AI hobbyists seeking accessible ML resources

Hugging Face powers some of the largest open-source libraries and is a trusted platform for AI professionals globally.

---

## Careers at Hugging Face

Join a passionate team shaping the future of AI. Hugging Face looks for individuals who:

- Are passionate about AI and open-source technologies
- Thrive in collaborative, fast-paced environments
- Want to impact the AI community positively with ethical innovation
- Possess strong engineering, research, or community-building skills

Careers span roles in software engineering, research science, community engagement, product development, and enterprise support.

---

## Get Involved

- **Explore AI apps and models:** Discover new ML capabilities at your fingertips.
- **Contribute:** Share your models, datasets, or applications with a global community.
- **Build your profile:** Showcase your work and collaborate openly.
- **Join the team:** Be part of a transformative AI journey.

**Visit:** [huggingface.co](https://huggingface.co)  
**Sign Up** for free to start exploring and contributing today!

---

### Brand Colors & Assets

- Primary Colors: #FFD21E (Yellow), #FF9D00 (Orange), #6B7280 (Gray)
- Logo available in SVG, PNG, AI formats for branding and collaboration purposes.

---

**Hugging Face – The AI community building the future.**  
Empowering open, ethical, and collaborative machine learning worldwide.

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>